# Flight Price Prediction using Machine Learning

**Course:** Machine Learning / Data Science  
**Dataset:** Indian Domestic Flight Prices (Clean_Dataset.csv)  
**Language:** Python 3  

| Name | Roll Number |
|---|---|
| Aradhya Mishra | 2423371 |
| Nishant Kumar | 2423349 |
| Harsh Mittal | 2423373 |
| Deshpande Rugved Shirish | 2423370 |

---

The goal of this project is to build a machine learning model that can predict the price of a domestic flight ticket in India. The dataset contains information about flights operated by six major Indian airlines across six cities, including features like departure time, number of stops, flight duration, travel class, and how many days in advance the ticket is being booked.

The project follows a standard ML pipeline: data loading, exploratory analysis, preprocessing, model training, evaluation, and conclusions.

---
## 1. Research Questions, Objectives and Hypotheses

### Research Questions

1. Can we reliably predict the price of a flight ticket based on features such as airline, route, class, stops, departure time, flight duration, and days left before departure?
2. Which features have the most influence on the ticket price?
3. Does booking a ticket earlier result in a meaningfully lower price?

### Objectives

- Perform exploratory data analysis to understand the structure and distributions of the data.
- Clean and preprocess the data for use in a machine learning model.
- Train and compare multiple regression models to predict flight prices.
- Evaluate model performance using standard regression metrics (MAE, RMSE, R2).
- Identify the features that most strongly influence flight pricing.

### Hypotheses

**Null Hypothesis (H0):**  
The features available in the dataset — airline, source city, destination city, departure time, number of stops, arrival time, travel class, flight duration, and days left before departure — have no statistically significant effect on the flight ticket price. A machine learning model built on these features will not predict prices better than simply guessing the mean price.

**Alternative Hypothesis (H1):**  
The above features do have a significant relationship with flight ticket prices, and a machine learning model trained on them will explain a substantial portion of the price variance, achieving an R2 score meaningfully greater than zero.

---

## 2. Importing Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

---
## 3. Loading the Dataset

The dataset used is `Clean_Dataset.csv`, a preprocessed version of raw flight booking data scraped from the EaseMyTrip website. It contains 300,153 records with 11 columns covering airline, route, timing, stops, class, duration, and price.

In [ ]:
# Load dataset -- make sure Clean_Dataset.csv is in the same directory as this notebook
df = pd.read_csv('Clean_Dataset.csv', index_col=0)

print(f'Shape of dataset: {df.shape[0]:,} rows x {df.shape[1]} columns')
df.head(10)

In [ ]:
df.info()

In [ ]:
df.describe(include='all')

---
## 4. Exploratory Data Analysis (EDA)

Before building any model, it is important to understand what the data looks like. This section covers checking for missing values and data quality issues, as well as visualising the relationships between the features and the target variable (price).

### 4.1 Missing Values and Data Quality

In [ ]:
missing = df.isnull().sum()
print('Missing values per column:')
print(missing)

if missing.sum() == 0:
    print('\nNo missing values found. The dataset is complete.')
else:
    print(f'\nTotal missing values: {missing.sum()}')

In [ ]:
duplicates = df.duplicated().sum()
print(f'Duplicate rows: {duplicates}')

print('\nColumn data types:')
print(df.dtypes)

In [ ]:
cat_cols = ['airline', 'source_city', 'destination_city',
            'departure_time', 'arrival_time', 'stops', 'class']

print('Unique values in categorical columns:\n')
for col in cat_cols:
    print(f'  {col}: {list(df[col].unique())}')

### 4.2 Distribution of Flight Prices

The price distribution is right-skewed. Most tickets are in the lower price range, but there is a long tail of expensive business class and last-minute bookings. The log-transformed version gives a clearer view of the distribution shape.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df['price'], bins=60, color='steelblue', edgecolor='white', alpha=0.85)
axes[0].set_title('Distribution of Flight Prices')
axes[0].set_xlabel('Ticket Price (INR)')
axes[0].set_ylabel('Number of Flights')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))

axes[1].hist(np.log1p(df['price']), bins=60, color='darkorange', edgecolor='white', alpha=0.85)
axes[1].set_title('Log-Transformed Price Distribution')
axes[1].set_xlabel('log(Price + 1)')
axes[1].set_ylabel('Number of Flights')

plt.suptitle('Flight Price Distribution Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print(f'Mean price:    Rs.{df["price"].mean():,.0f}')
print(f'Median price:  Rs.{df["price"].median():,.0f}')
print(f'Minimum price: Rs.{df["price"].min():,.0f}')
print(f'Maximum price: Rs.{df["price"].max():,.0f}')
print(f'Std deviation: Rs.{df["price"].std():,.0f}')

### 4.3 Price by Airline

Different airlines follow very different pricing strategies. Vistara and Air India are full-service carriers that charge considerably more, while IndiGo, SpiceJet, and AirAsia operate as low-cost carriers.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

airline_avg = df.groupby('airline')['price'].mean().sort_values(ascending=False)
bars = axes[0].bar(airline_avg.index, airline_avg.values,
                   color=sns.color_palette('muted', len(airline_avg)), edgecolor='black')
axes[0].set_title('Average Ticket Price by Airline')
axes[0].set_xlabel('Airline')
axes[0].set_ylabel('Average Price (INR)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 200,
                 f'Rs.{bar.get_height()/1000:.1f}K', ha='center', fontsize=8.5)
axes[0].tick_params(axis='x', rotation=25)

airlines_order = df.groupby('airline')['price'].median().sort_values(ascending=False).index
sns.boxplot(data=df, x='airline', y='price', order=airlines_order, ax=axes[1], palette='muted')
axes[1].set_title('Price Spread by Airline (Boxplot)')
axes[1].set_xlabel('Airline')
axes[1].set_ylabel('Price (INR)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
axes[1].tick_params(axis='x', rotation=25)

plt.tight_layout()
plt.show()

### 4.4 Price by Travel Class

Travel class is expected to be one of the strongest predictors of price. Business class fares are substantially higher than economy fares across all airlines and routes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

class_counts = df['class'].value_counts()
axes[0].pie(class_counts.values, labels=class_counts.index,
            autopct='%1.1f%%', colors=['#4C72B0', '#DD8452'],
            startangle=90, wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[0].set_title('Proportion of Tickets by Class')

for cls, color in zip(['Economy', 'Business'], ['#4C72B0', '#DD8452']):
    subset = df[df['class'] == cls]['price']
    axes[1].hist(subset, bins=50, alpha=0.65, label=cls, color=color, edgecolor='white')
axes[1].set_title('Price Distribution: Economy vs Business')
axes[1].set_xlabel('Ticket Price (INR)')
axes[1].set_ylabel('Frequency')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
axes[1].legend()

plt.tight_layout()
plt.show()

print('Average price by class:')
print(df.groupby('class')['price'].describe().round(0))

### 4.5 Price by Number of Stops

The relationship between stops and price is not straightforward. Non-stop flights are often more expensive since they offer convenience, but connecting flights booked at the last minute can also be very costly.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

stops_order = ['zero', 'one', 'two_or_more']
stop_labels = {'zero': 'Non-stop', 'one': '1 Stop', 'two_or_more': '2+ Stops'}
df['stops_label'] = df['stops'].map(stop_labels)

stop_avg = df.groupby('stops')['price'].mean().reindex(stops_order)
stop_avg.index = [stop_labels[s] for s in stop_avg.index]
bars = axes[0].bar(stop_avg.index, stop_avg.values,
                   color=['#4CAF50', '#FF9800', '#F44336'], edgecolor='black')
for bar in bars:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
                 f'Rs.{bar.get_height()/1000:.1f}K', ha='center', fontsize=9)
axes[0].set_title('Average Price by Number of Stops')
axes[0].set_xlabel('Stops')
axes[0].set_ylabel('Average Price (INR)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))

sns.boxplot(data=df, x='stops_label', y='price',
            order=['Non-stop', '1 Stop', '2+ Stops'],
            palette=['#4CAF50', '#FF9800', '#F44336'], ax=axes[1])
axes[1].set_title('Price Spread by Number of Stops')
axes[1].set_xlabel('Stops')
axes[1].set_ylabel('Price (INR)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))

plt.tight_layout()
plt.show()

df.drop(columns=['stops_label'], inplace=True)

### 4.6 Price vs Days Left Before Departure

The `days_left` column records how many days before departure the fare was observed. This is one of the most practically important features — prices generally rise sharply as the travel date approaches.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

days_avg = df.groupby('days_left')['price'].mean()
axes[0].plot(days_avg.index, days_avg.values, color='steelblue', linewidth=2)
axes[0].fill_between(days_avg.index, days_avg.values, alpha=0.15, color='steelblue')
axes[0].set_title('Average Price vs Days Before Departure')
axes[0].set_xlabel('Days Left (higher = booked further in advance)')
axes[0].set_ylabel('Average Price (INR)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))

sample = df.sample(5000, random_state=42)
axes[1].scatter(sample['days_left'], sample['price'], alpha=0.25, s=10, color='coral')
axes[1].set_title('Price vs Days Left (sample of 5,000 records)')
axes[1].set_xlabel('Days Left')
axes[1].set_ylabel('Price (INR)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))

plt.tight_layout()
plt.show()

### 4.7 Price by Route

In [ ]:
df['route'] = df['source_city'] + ' to ' + df['destination_city']
route_avg = df.groupby('route')['price'].mean().sort_values(ascending=False).head(15)

fig, ax = plt.subplots(figsize=(13, 6))
bars = ax.barh(route_avg.index[::-1], route_avg.values[::-1],
               color=sns.color_palette('Blues_d', len(route_avg)), edgecolor='black')
ax.set_title('Top 15 Routes by Average Ticket Price')
ax.set_xlabel('Average Price (INR)')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
for bar in bars:
    ax.text(bar.get_width() + 100, bar.get_y() + bar.get_height()/2,
            f'Rs.{bar.get_width()/1000:.1f}K', va='center', fontsize=8.5)
plt.tight_layout()
plt.show()

df.drop(columns=['route'], inplace=True)

### 4.8 Price by Departure Time

In [ ]:
time_order = ['Early_Morning', 'Morning', 'Afternoon', 'Evening', 'Night', 'Late_Night']
time_avg = df.groupby('departure_time')['price'].mean().reindex(time_order)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(time_avg.index, time_avg.values,
              color=sns.color_palette('coolwarm', len(time_avg)), edgecolor='black')
ax.set_title('Average Ticket Price by Departure Time Slot')
ax.set_xlabel('Departure Time')
ax.set_ylabel('Average Price (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
for bar in bars:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 100,
            f'Rs.{bar.get_height()/1000:.1f}K', ha='center', fontsize=9)
plt.tight_layout()
plt.show()

### 4.9 Price vs Flight Duration

In [ ]:
sample = df.sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(11, 5))
scatter = ax.scatter(sample['duration'], sample['price'],
                     c=sample['price'], cmap='viridis', alpha=0.4, s=15)
plt.colorbar(scatter, ax=ax, label='Price (INR)')
ax.set_title('Ticket Price vs Flight Duration')
ax.set_xlabel('Duration (hours)')
ax.set_ylabel('Price (INR)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
plt.tight_layout()
plt.show()

### 4.10 Correlation Heatmap

The heatmap below shows correlations between the numerical features. Most features in this dataset are categorical, so a deeper correlation analysis is carried out after encoding in the preprocessing step.

In [ ]:
num_cols = ['duration', 'days_left', 'price']
corr_matrix = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(6, 4))
sns.heatmap(corr_matrix, annot=True, fmt='.3f', cmap='coolwarm',
            linewidths=0.5, ax=ax, vmin=-1, vmax=1)
ax.set_title('Correlation Heatmap (Numerical Features)')
plt.tight_layout()
plt.show()

---
## 5. Data Preprocessing

Before training a model, the dataset needs to be prepared. This involves removing irrelevant columns, encoding categorical variables into numeric form, splitting into training and test sets, and scaling the continuous features.

### 5.1 Data Wrangling

In [ ]:
# Work on a copy to keep the original dataframe intact
df_model = df.copy()

# The 'flight' column contains individual flight codes (e.g. SG-8709).
# With thousands of unique values that don't generalise across airlines,
# it would cause the model to overfit. Dropping it.
df_model.drop(columns=['flight'], inplace=True)

print('Columns retained for modelling:')
print(list(df_model.columns))
print(f'\nShape: {df_model.shape}')

### 5.2 Encoding Categorical Variables

Machine learning models require numerical input. Categorical columns are encoded as follows:
- **Ordinal encoding** is used for features with a natural order (stops, class, departure/arrival time).
- **Label encoding** is used for nominal categories with no inherent order (airline, source city, destination city).

In [ ]:
# Stops: ordered zero < one < two_or_more
stops_map = {'zero': 0, 'one': 1, 'two_or_more': 2}
df_model['stops'] = df_model['stops'].map(stops_map)

# Travel class: binary
class_map = {'Economy': 0, 'Business': 1}
df_model['class'] = df_model['class'].map(class_map)

# Time slots: ordered by time of day
time_map = {
    'Early_Morning': 0,
    'Morning':       1,
    'Afternoon':     2,
    'Evening':       3,
    'Night':         4,
    'Late_Night':    5
}
df_model['departure_time'] = df_model['departure_time'].map(time_map)
df_model['arrival_time']   = df_model['arrival_time'].map(time_map)

# Airline, source city, destination city: no natural order -- use label encoding
le = LabelEncoder()
for col in ['airline', 'source_city', 'destination_city']:
    df_model[col] = le.fit_transform(df_model[col])

print('Encoding complete. Sample of encoded data:')
df_model.head()

### 5.3 Feature Selection and Train/Test Split

The dataset is split 80/20 into training and test sets. The test set is held out and only used at evaluation time -- the model never sees this data during training.

In [ ]:
X = df_model.drop(columns=['price'])
y = df_model['price']

print(f'Features ({X.shape[1]} total): {list(X.columns)}')
print(f'Target: price')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f'\nTraining set:  {X_train.shape[0]:,} records')
print(f'Test set:      {X_test.shape[0]:,} records')

### 5.4 Feature Scaling

Standard scaling (zero mean, unit variance) is applied to the continuous numerical columns `duration` and `days_left`. The scaler is fit only on training data and then applied to the test set to prevent data leakage.

In [ ]:
scaler = StandardScaler()
scale_cols = ['duration', 'days_left']

X_train[scale_cols] = scaler.fit_transform(X_train[scale_cols])
X_test[scale_cols]  = scaler.transform(X_test[scale_cols])

print('Scaling applied to:', scale_cols)
print('\nTraining set after scaling (duration and days_left):')
print(X_train[scale_cols].describe().round(3))

### 5.5 Final Feature Set Summary

In [ ]:
feature_descriptions = {
    'airline':           'Airline carrier (label encoded, 6 carriers)',
    'source_city':       'City of departure (label encoded, 6 cities)',
    'departure_time':    'Departure time slot (ordinal 0-5: Early Morning to Late Night)',
    'stops':             'Number of stops (0 = non-stop, 1 = one stop, 2 = two or more)',
    'arrival_time':      'Arrival time slot (same ordinal scale as departure_time)',
    'destination_city':  'City of arrival (label encoded, 6 cities)',
    'class':             'Travel class (0 = Economy, 1 = Business)',
    'duration':          'Total flight duration in hours (standard scaled)',
    'days_left':         'Days before departure when fare was observed (standard scaled)'
}

feat_summary = pd.DataFrame({
    'Feature':     list(feature_descriptions.keys()),
    'Description': list(feature_descriptions.values())
})
print('Final feature set used for training:\n')
print(feat_summary.to_string(index=False))
print(f'\nTotal features: {X_train.shape[1]}')
print(f'Training size:  {X_train.shape[0]:,}')
print(f'Test size:      {X_test.shape[0]:,}')

---
## 6. Model Building

### 6.1 Model Choice and Reasoning

Three models are trained and compared:

| Model | Why it was chosen |
|---|---|
| Linear Regression | Acts as a simple baseline. Shows how much improvement non-linear models bring. |
| Random Forest Regressor | An ensemble of decision trees. Handles non-linear relationships and feature interactions effectively without much hyperparameter tuning. |
| Gradient Boosting Regressor | Trains trees sequentially, each correcting the errors of the previous one. Typically very strong on structured tabular data. |

The final model is selected based on R2 and RMSE on the held-out test set.

### 6.2 Training the Models

In [ ]:
# Model 1: Linear Regression (baseline)
print('Training Linear Regression...')
lr = LinearRegression()
lr.fit(X_train, y_train)
print('Done.')

# Model 2: Random Forest
# 100 trees, depth capped at 15 to limit overfitting on this large dataset
print('\nTraining Random Forest Regressor (may take a minute)...')
rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=15,
    min_samples_split=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train, y_train)
print('Done.')

# Model 3: Gradient Boosting
print('\nTraining Gradient Boosting Regressor (may take a minute)...')
gb = GradientBoostingRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)
gb.fit(X_train, y_train)
print('Done.')

### 6.3 Random Forest Architecture Diagram

The diagram below illustrates how a Random Forest works. Multiple decision trees are trained independently on random subsets of the data. Their individual predictions are then averaged to produce the final output. Averaging across many trees reduces overfitting and improves generalisation.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))
ax.set_xlim(0, 10)
ax.set_ylim(0, 6)
ax.axis('off')
ax.set_facecolor('#f7f7f7')
fig.patch.set_facecolor('#f7f7f7')

ax.text(5, 5.6, 'Random Forest Regressor: How It Works',
        ha='center', va='center', fontsize=13, fontweight='bold')

# Input block
rect = plt.Rectangle((0.1, 2.4), 1.6, 1.7, linewidth=1.5,
                       edgecolor='#333333', facecolor='#AED6F1')
ax.add_patch(rect)
ax.text(0.9, 3.25, 'Input\nFeatures\n(9 columns)', ha='center', va='center', fontsize=9)

ax.annotate('', xy=(2.1, 3.25), xytext=(1.7, 3.25),
            arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))

# Decision trees
tree_data = [(2.4, 'Tree 1', '#A9DFBF'), (3.9, 'Tree 2', '#F9E79F'), (5.3, 'Tree 100', '#F5CBA7')]
for i, (xpos, label, color) in enumerate(tree_data):
    rect = plt.Rectangle((xpos, 2.0), 1.2, 2.4, linewidth=1.5,
                           edgecolor='#333333', facecolor=color)
    ax.add_patch(rect)
    ax.text(xpos + 0.6, 3.25, f'Decision\n{label}\n(trained on\nrandom subset)',
            ha='center', va='center', fontsize=7.5)

ax.text(4.7, 3.25, '...', ha='center', va='center', fontsize=16)

# Arrow to averaging
ax.annotate('', xy=(7.2, 3.25), xytext=(6.6, 3.25),
            arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))

# Averaging block
rect = plt.Rectangle((7.2, 2.5), 1.6, 1.5, linewidth=1.5,
                       edgecolor='#333333', facecolor='#D7BDE2')
ax.add_patch(rect)
ax.text(8.0, 3.25, 'Average of\nall tree\npredictions', ha='center', va='center', fontsize=9)

# Arrow to output
ax.annotate('', xy=(9.0, 3.25), xytext=(8.8, 3.25),
            arrowprops=dict(arrowstyle='->', color='#333333', lw=1.5))

# Output
rect = plt.Rectangle((9.0, 2.7), 0.9, 1.1, linewidth=1.5,
                       edgecolor='#333333', facecolor='#FADBD8')
ax.add_patch(rect)
ax.text(9.45, 3.25, 'Predicted\nPrice', ha='center', va='center', fontsize=9)

plt.tight_layout()
plt.show()

---
## 7. Model Testing and Results

### 7.1 Evaluating All Three Models

Each model is evaluated on the test set using three metrics:
- **MAE (Mean Absolute Error):** Average absolute difference between predicted and actual prices. Easy to interpret in the original currency.
- **RMSE (Root Mean Squared Error):** Similar to MAE but penalises large errors more heavily.
- **R2 Score:** Proportion of variance in price explained by the model. 1.0 is perfect; 0.0 means it does no better than predicting the mean.

In [ ]:
def evaluate_model(name, model, X_tr, X_te, y_tr, y_te):
    y_pred_te = model.predict(X_te)
    y_pred_tr = model.predict(X_tr)

    mae      = mean_absolute_error(y_te, y_pred_te)
    rmse     = np.sqrt(mean_squared_error(y_te, y_pred_te))
    r2       = r2_score(y_te, y_pred_te)
    r2_train = r2_score(y_tr, y_pred_tr)

    print(f'{name}')
    print(f'  MAE:         Rs.{mae:,.0f}')
    print(f'  RMSE:        Rs.{rmse:,.0f}')
    print(f'  R2 (test):   {r2:.4f}')
    print(f'  R2 (train):  {r2_train:.4f}')
    print()

    return {
        'Model': name, 'MAE': mae, 'RMSE': rmse,
        'R2_Test': r2, 'R2_Train': r2_train,
        'Predictions': y_pred_te
    }

results = []
results.append(evaluate_model('Linear Regression', lr, X_train, X_test, y_train, y_test))
results.append(evaluate_model('Random Forest',     rf, X_train, X_test, y_train, y_test))
results.append(evaluate_model('Gradient Boosting', gb, X_train, X_test, y_train, y_test))

In [ ]:
summary = pd.DataFrame([
    {
        'Model':      r['Model'],
        'MAE (Rs.)':  f"{r['MAE']:,.0f}",
        'RMSE (Rs.)': f"{r['RMSE']:,.0f}",
        'R2 Test':    f"{r['R2_Test']:.4f}",
        'R2 Train':   f"{r['R2_Train']:.4f}"
    }
    for r in results
])
summary.set_index('Model', inplace=True)
print('Model Comparison Summary:')
summary

### 7.2 Model Performance Comparison Chart

In [ ]:
model_names = [r['Model'] for r in results]
maes  = [r['MAE']     for r in results]
rmses = [r['RMSE']    for r in results]
r2s   = [r['R2_Test'] for r in results]

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
palette = ['#4C72B0', '#55A868', '#C44E52']

bars = axes[0].bar(model_names, maes, color=palette, edgecolor='black')
axes[0].set_title('Mean Absolute Error (lower is better)')
axes[0].set_ylabel('MAE (INR)')
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
for bar, val in zip(bars, maes):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'Rs.{val/1000:.1f}K', ha='center', fontsize=9)
axes[0].tick_params(axis='x', rotation=15)

bars = axes[1].bar(model_names, rmses, color=palette, edgecolor='black')
axes[1].set_title('Root Mean Squared Error (lower is better)')
axes[1].set_ylabel('RMSE (INR)')
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
for bar, val in zip(bars, rmses):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                 f'Rs.{val/1000:.1f}K', ha='center', fontsize=9)
axes[1].tick_params(axis='x', rotation=15)

bars = axes[2].bar(model_names, r2s, color=palette, edgecolor='black')
axes[2].set_title('R2 Score (higher is better)')
axes[2].set_ylabel('R2 Score')
axes[2].set_ylim(0, 1.08)
for bar, val in zip(bars, r2s):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.008,
                 f'{val:.3f}', ha='center', fontsize=9)
axes[2].tick_params(axis='x', rotation=15)

plt.suptitle('Model Performance on Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 7.3 Actual vs Predicted: Line of Best Fit

The scatter plot shows actual prices on the x-axis and predicted prices on the y-axis for the Random Forest model. Points on the dashed red line represent perfect predictions. The closer the point cloud is to this line, the better the model is performing.

In [ ]:
y_pred_rf  = rf.predict(X_test)
y_test_arr = np.array(y_test)

rng = np.random.default_rng(42)
idx = rng.choice(len(y_test_arr), size=3000, replace=False)

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Actual vs Predicted
axes[0].scatter(y_test_arr[idx], y_pred_rf[idx], alpha=0.3, s=10, color='steelblue')
lo = min(y_test_arr.min(), y_pred_rf.min())
hi = max(y_test_arr.max(), y_pred_rf.max())
axes[0].plot([lo, hi], [lo, hi], color='red', linewidth=1.8,
             linestyle='--', label='Perfect prediction line')
axes[0].set_title('Random Forest: Actual vs Predicted Price')
axes[0].set_xlabel('Actual Price (INR)')
axes[0].set_ylabel('Predicted Price (INR)')
axes[0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
axes[0].legend()

# Residual plot
residuals = y_test_arr - y_pred_rf
axes[1].scatter(y_pred_rf[idx], residuals[idx], alpha=0.3, s=10, color='darkorange')
axes[1].axhline(y=0, color='black', linewidth=1.5, linestyle='--', label='Zero error line')
axes[1].set_title('Residual Plot (Actual minus Predicted)')
axes[1].set_xlabel('Predicted Price (INR)')
axes[1].set_ylabel('Residual (INR)')
axes[1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
axes[1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
axes[1].legend()

plt.tight_layout()
plt.show()

print('Error statistics for Random Forest on the test set:')
print(f'  Mean residual:          Rs.{residuals.mean():,.0f}  (should be close to 0)')
print(f'  Std of residuals:       Rs.{residuals.std():,.0f}')
print(f'  Largest overestimate:   Rs.{(-residuals).max():,.0f}')
print(f'  Largest underestimate:  Rs.{residuals.max():,.0f}')

### 7.4 Error Distribution

A well-behaved regression model should produce residuals that are approximately normally distributed and centred near zero. A distribution that is skewed or offset suggests the model has systematic bias in one direction.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(residuals, bins=60, color='steelblue', edgecolor='white', alpha=0.85)
ax.axvline(x=0, color='red', linewidth=2, linestyle='--', label='Zero error')
ax.axvline(x=residuals.mean(), color='orange', linewidth=2, linestyle='-',
           label=f'Mean residual (Rs.{residuals.mean():,.0f})')
ax.set_title('Distribution of Prediction Errors (Residuals) -- Random Forest')
ax.set_xlabel('Residual: Actual minus Predicted (INR)')
ax.set_ylabel('Frequency')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'Rs.{int(x/1000)}K'))
ax.legend()
plt.tight_layout()
plt.show()

---
## 8. Feature Importance

Random Forest provides a built-in measure of how much each feature contributed to reducing prediction error across all trees. A higher importance score means that feature was more frequently and effectively used to split the data into accurate predictions.

In [ ]:
importances = rf.feature_importances_
feat_imp    = pd.Series(importances, index=X.columns).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
colors_fi = sns.color_palette('viridis', len(feat_imp))
bars = ax.barh(feat_imp.index, feat_imp.values, color=colors_fi, edgecolor='black')
ax.set_title('Feature Importance -- Random Forest Regressor')
ax.set_xlabel('Importance Score')
for bar, val in zip(bars, feat_imp.values):
    ax.text(bar.get_width() + 0.001, bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print('Feature importance ranking (highest to lowest):')
for feat, imp in feat_imp.sort_values(ascending=False).items():
    bar_vis = '#' * int(imp * 100)
    print(f'  {feat:22s}: {imp:.4f}  {bar_vis}')

---
## 9. Conclusions

### Final Model Performance

In [ ]:
final_mae  = mean_absolute_error(y_test, y_pred_rf)
final_rmse = np.sqrt(mean_squared_error(y_test, y_pred_rf))
final_r2   = r2_score(y_test, y_pred_rf)

print('=' * 55)
print('  FINAL RESULTS -- Random Forest Regressor')
print('=' * 55)
print(f'  Dataset size:         {df.shape[0]:,} records')
print(f'  Features used:        {X.shape[1]}')
print(f'  Train / Test split:   80% / 20%')
print(f'  MAE:                  Rs.{final_mae:,.0f}')
print(f'  RMSE:                 Rs.{final_rmse:,.0f}')
print(f'  R2 Score (test set):  {final_r2:.4f}')
print('=' * 55)

### Hypothesis Outcome

Based on the results, the **null hypothesis (H0) is rejected**. The features in the dataset are collectively strong predictors of flight ticket price. The Random Forest model achieved an R2 score well above 0.0, which means the model explains a significant proportion of the variance in ticket prices -- far more than could be attributed to chance.

### Key Observations

**Travel class** turned out to be the single strongest predictor of price. The gap between economy and business fares is very large across all airlines and routes. This is expected, and any model that ignores class would perform poorly.

**Days left before departure** was the second most influential feature. Prices tend to be lower when tickets are booked well in advance, and rise considerably as the departure date approaches. This is consistent with standard airline dynamic pricing behaviour.

**Airline** had a meaningful effect on price. Full-service carriers like Vistara and Air India charge significantly more than low-cost carriers like SpiceJet and IndiGo for comparable routes.

**Flight duration** and **number of stops** also contributed to predictions, though less so than class and booking timing. Longer routes and connecting flights introduce more variability in pricing.

**Departure time** had a relatively small but non-zero effect. Early morning and late-night flights tend to be slightly cheaper on average, possibly because they are less convenient and thus less in demand.

### Model Comparison

Linear Regression performed the weakest of the three. This is expected given the clearly non-linear relationships in the data, particularly the sharp interaction between class and price. Both ensemble methods substantially outperformed the baseline. Random Forest and Gradient Boosting achieved similar results; Random Forest was selected as the final model due to slightly better performance and simpler tuning requirements.

### Practical Implications

For travellers, this analysis confirms that booking in advance and choosing economy class are the two most effective ways to reduce the cost of a domestic flight. Route and airline choice also matter, but to a lesser degree. A model like this could be integrated into a travel platform to give passengers a price estimate before they complete a search, or used to flag fares that appear anomalously priced relative to similar bookings.